In [33]:

import pandas as pd
import sys
from pathlib import Path

df = pd.read_csv("../data/processed/cp_processed.csv")



In [38]:
df.columns

Index(['Program', 'Year', 'Period', 'Week Ending', 'Geo Level', 'State',
       'State ANSI', 'Ag District', 'Ag District Code', 'County',
       'County ANSI', 'Zip Code', 'Region', 'watershed_code', 'Watershed',
       'Commodity', 'Data Item', 'Domain', 'Domain Category', 'Value',
       'CV (%)'],
      dtype='str')

In [39]:
for commodity in sorted(df["Commodity"].dropna().unique()):
    print(f"\n{commodity}")
    print(
        df.loc[
            df["Commodity"] == commodity,
            "Data Item",
        ].dropna().unique().tolist()
    )


CORN
['CORN - PROGRESS, MEASURED IN PCT PLANTED', 'CORN - PROGRESS, MEASURED IN PCT EMERGED', 'CORN - PROGRESS, MEASURED IN PCT SILKING', 'CORN - PROGRESS, MEASURED IN PCT DOUGH', 'CORN - PROGRESS, MEASURED IN PCT DENTED', 'CORN - PROGRESS, MEASURED IN PCT MATURE', 'CORN - PROGRESS, MEASURED IN PCT MILK']

SOYBEANS
['SOYBEANS - PROGRESS, MEASURED IN PCT PLANTED', 'SOYBEANS - PROGRESS, MEASURED IN PCT EMERGED', 'SOYBEANS - PROGRESS, MEASURED IN PCT BLOOMING', 'SOYBEANS - PROGRESS, MEASURED IN PCT SETTING PODS', 'SOYBEANS - PROGRESS, MEASURED IN PCT COLORING', 'SOYBEANS - PROGRESS, MEASURED IN PCT DROPPING LEAVES', 'SOYBEANS - PROGRESS, MEASURED IN PCT MATURE', 'SOYBEANS - PROGRESS, MEASURED IN PCT FULLY PODDED', 'SOYBEANS - PROGRESS, MEASURED IN PCT HARVESTED']

WHEAT
['WHEAT, WINTER - PROGRESS, MEASURED IN PCT PLANTED', 'WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED', 'WHEAT, SPRING, DURUM - PROGRESS, MEASURED IN PCT EMERGED', 'WHEAT, SPRING, DURUM - PROGRESS, MEASURED IN PCT PLAN

In [40]:
pd.crosstab(
    df["Commodity"],
    df["Geo Level"],
)

Geo Level,NATIONAL,STATE
Commodity,,
CORN,544,22073
SOYBEANS,596,22754
WHEAT,868,26009


In [46]:
import sys
from pathlib import Path

from src.loaders.load_nass import load_cp
from src.transforms.transform_cp import prepare_cp_data

ROOT_DIR = Path.cwd()

while not (ROOT_DIR / "src").exists():
    if ROOT_DIR.parent == ROOT_DIR:
        raise FileNotFoundError("Could not find project root")
    ROOT_DIR = ROOT_DIR.parent

sys.path.insert(0, str(ROOT_DIR))

df = load_cp()

cp_data = prepare_cp_data(df)

national_df = cp_data["national"]
state_df = cp_data["state"]


print("\nNATIONAL DATA")
print(national_df.shape)

print("\nSTATE DATA")
print(state_df.shape)


print("\nNATIONAL COMMODITY AND CROP TYPE")
print(
    national_df.groupby(
        ["Commodity", "Crop Type"]
    ).size()
)


print("\nNATIONAL MEASURE SUMMARY")
print(
    national_df.groupby(
        ["Commodity", "Crop Type", "Measure"]
    ).agg(
        Rows=("Value", "size"),
        Years=("Year", "nunique"),
        Weeks=("Week", "nunique"),
        Five_Year_Avg=("5-Year Avg", "count"),
    ).to_string()
)


print("\nSTATE COMMODITY AND CROP TYPE")
print(
    state_df.groupby(
        ["Commodity", "Crop Type"]
    ).size()
)


print("\nMISSING VALUES IN NATIONAL DATA")
print(
    national_df[
        [
            "Commodity",
            "Crop Type",
            "Measure",
            "Year",
            "Week",
            "5-Year Avg",
            "5-Year Min",
            "5-Year Max",
        ]
    ].isna().sum()
)


2026-11-10 09:29:14.958 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-10 09:29:14.961 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-10 09:29:14.961 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.



NATIONAL DATA
(2008, 27)

STATE DATA
(70836, 24)

NATIONAL COMMODITY AND CROP TYPE
Commodity  Crop Type                 
CORN       Corn                          544
SOYBEANS   Soybeans                      596
WHEAT      Spring Wheat (Excl. Durum)    368
           Winter Wheat                  500
dtype: int64

NATIONAL MEASURE SUMMARY
                                                      Rows  Years  Weeks  Five_Year_Avg
Commodity Crop Type                  Measure                                           
CORN      Corn                       Dented             86     10     11             36
                                     Dough              93     10     12             38
                                     Emerged            93     10     13             36
                                     Mature             91     10     12             34
                                     Planted           100     10     13             36
                                     Silkin

In [49]:
corn_planted = national_df.loc[
    (national_df["Commodity"] == "CORN")
    & (national_df["Measure"] == "Planted")
].copy()

print(
    corn_planted.pivot_table(
        index="Week",
        columns="Year",
        values="Value",
        aggfunc="mean",
    ).to_string()
)

Year  2017  2018  2019  2020  2021  2022  2023  2024  2025  2026
Week                                                            
13     NaN   NaN   NaN   NaN   2.0   2.0   2.0   2.0   NaN   NaN
14     3.0   2.0   2.0   NaN   4.0   2.0   3.0   3.0   2.0   3.0
15     6.0   3.0   3.0   3.0   8.0   4.0   8.0   6.0   4.0   5.0
16    17.0   5.0   6.0   7.0  17.0   7.0  14.0  12.0  12.0  11.0
17    34.0  17.0  15.0  27.0  46.0  14.0  26.0  27.0  24.0  25.0
18    47.0  39.0  23.0  51.0  67.0  22.0  49.0  36.0  40.0  38.0
19    71.0  62.0  30.0  67.0  80.0  49.0  65.0  49.0  62.0  57.0
20    84.0  81.0  49.0  80.0  90.0  72.0  81.0  70.0  78.0  76.0
21    91.0  92.0  58.0  88.0  95.0  86.0  92.0  83.0  87.0  86.0
22    96.0  97.0  67.0  93.0   NaN  94.0  96.0  91.0  93.0  93.0
23     NaN   NaN  83.0  97.0   NaN  97.0   NaN  95.0  97.0  97.0
24     NaN   NaN  92.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN
25     NaN   NaN  96.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN


In [48]:
print(
    national_df.groupby(
        ["Commodity", "Crop Type", "Measure"]
    ).agg(
        Total=("Value", "count"),
        Benchmark=("5-Year Avg", "count"),
    ).assign(
        Coverage=lambda x: (
            x["Benchmark"] / x["Total"] * 100
        ).round(1)
    ).to_string()
)

                                                      Total  Benchmark  Coverage
Commodity Crop Type                  Measure                                    
CORN      Corn                       Dented              86         36      41.9
                                     Dough               93         38      40.9
                                     Emerged             93         36      38.7
                                     Mature              91         34      37.4
                                     Planted            100         36      36.0
                                     Silking             81         29      35.8
SOYBEANS  Soybeans                   Blooming            98         43      43.9
                                     Dropping Leaves     89         32      36.0
                                     Emerged             93         39      41.9
                                     Harvested           98         33      33.7
                            

In [54]:
df = load_cp()

prefixes = (
    df["Data Item"]
    .astype("string")
    .str.extract(
        r"^(.*?)\s*-\s*PROGRESS",
        flags=2,
        expand=False,
    )
    .dropna()
    .str.strip()
    .str.upper()
    .unique()
)

print(sorted(prefixes))

['CORN', 'SOYBEANS', 'WHEAT, SPRING, (EXCL DURUM)', 'WHEAT, SPRING, DURUM', 'WHEAT, WINTER']


In [55]:
df = load_cp()

prefixes = (
    df["Data Item"]
    .astype("string")
    .str.extract(
        r"^(.*?)\s*-\s*PROGRESS",
        flags=2,
        expand=False,
    )
    .dropna()
    .str.strip()
    .str.upper()
    .unique()
)

print(sorted(prefixes))

['CORN', 'SOYBEANS', 'WHEAT, SPRING, (EXCL DURUM)', 'WHEAT, SPRING, DURUM', 'WHEAT, WINTER']


In [52]:
corn_planted = national_df.loc[
    (national_df["Commodity"] == "CORN")
    & (national_df["Measure"] == "Planted")
].copy()

print(
    corn_planted.pivot_table(
        index="Week",
        columns="Year",
        values="Value",
        aggfunc="mean",
    ).to_string()
)

Year  2017  2018  2019  2020  2021  2022  2023  2024  2025  2026
Week                                                            
13     NaN   NaN   NaN   NaN   2.0   2.0   2.0   2.0   NaN   NaN
14     3.0   2.0   2.0   NaN   4.0   2.0   3.0   3.0   2.0   3.0
15     6.0   3.0   3.0   3.0   8.0   4.0   8.0   6.0   4.0   5.0
16    17.0   5.0   6.0   7.0  17.0   7.0  14.0  12.0  12.0  11.0
17    34.0  17.0  15.0  27.0  46.0  14.0  26.0  27.0  24.0  25.0
18    47.0  39.0  23.0  51.0  67.0  22.0  49.0  36.0  40.0  38.0
19    71.0  62.0  30.0  67.0  80.0  49.0  65.0  49.0  62.0  57.0
20    84.0  81.0  49.0  80.0  90.0  72.0  81.0  70.0  78.0  76.0
21    91.0  92.0  58.0  88.0  95.0  86.0  92.0  83.0  87.0  86.0
22    96.0  97.0  67.0  93.0   NaN  94.0  96.0  91.0  93.0  93.0
23     NaN   NaN  83.0  97.0   NaN  97.0   NaN  95.0  97.0  97.0
24     NaN   NaN  92.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN
25     NaN   NaN  96.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN


In [73]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path(
    "/Users/jason/Desktop/Intern/agri_dashboard/data/raw/nass"
)

files = sorted(RAW_DIR.glob("*.csv"))

for path in files:
    df = pd.read_csv(path, low_memory=False)

    print(f"\n{'=' * 60}")
    print(path.name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    for column in ["Commodity", "Data Item", "Year"]:
        if column in df.columns:
            print(f"\n{column}:")
            print(df[column].dropna().unique()[:15])


cc_corn.csv
Shape: (46430, 21)
Columns: ['Program', 'Year', 'Period', 'Week Ending', 'Geo Level', 'State', 'State ANSI', 'Ag District', 'Ag District Code', 'County', 'County ANSI', 'Zip Code', 'Region', 'watershed_code', 'Watershed', 'Commodity', 'Data Item', 'Domain', 'Domain Category', 'Value', 'CV (%)']

Commodity:
<ArrowStringArray>
['CORN']
Length: 1, dtype: str

Data Item:
<ArrowStringArray>
['CORN - CONDITION, MEASURED IN PCT EXCELLENT',
      'CORN - CONDITION, MEASURED IN PCT FAIR',
      'CORN - CONDITION, MEASURED IN PCT GOOD',
      'CORN - CONDITION, MEASURED IN PCT POOR',
 'CORN - CONDITION, MEASURED IN PCT VERY POOR']
Length: 5, dtype: str

Year:
[2026 2025 2024 2023 2022 2021 2020 2019 2018 2017]

cc_soy.csv
Shape: (35870, 21)
Columns: ['Program', 'Year', 'Period', 'Week Ending', 'Geo Level', 'State', 'State ANSI', 'Ag District', 'Ag District Code', 'County', 'County ANSI', 'Zip Code', 'Region', 'watershed_code', 'Watershed', 'Commodity', 'Data Item', 'Domain', 'Domain

In [74]:
from src.loaders.load_nass import load_cp

df = load_cp()

cp_data = module.prepare_cp_base_data(df)

calendar_df = module.prepare_cp_calendar_data(
    cp_data,
    commodity="CORN",
    crop_type="Corn",
)

calendar_df

NameError: name 'module' is not defined

In [78]:
import pandas as pd
from pathlib import Path

project_path = Path(
    "/Users/jason/Desktop/Intern/agri_dashboard"
)

if not project_path.exists():
    raise FileNotFoundError(
        f"Project folder not found: {project_path}"
    )

files = [
    file
    for file in project_path.rglob("*cp_processed*")
    if file.is_file()
    and file.suffix.lower() in [".csv", ".parquet"]
]

if not files:
    raise FileNotFoundError(
        "Could not find cp_processed CSV or Parquet file "
        "inside the project folder."
    )

print("=" * 80)
print("FILES FOUND")
print("=" * 80)

for i, file in enumerate(files, start=1):
    print(f"{i}. {file}")

if len(files) > 1:
    print("\nMultiple files found. Using the most recently modified file.")

file_path = max(
    files,
    key=lambda file: file.stat().st_mtime,
)

print("\n" + "=" * 80)
print("LOADING DATA")
print("=" * 80)

if file_path.suffix.lower() == ".csv":
    cp_processed = pd.read_csv(file_path)
elif file_path.suffix.lower() == ".parquet":
    cp_processed = pd.read_parquet(file_path)

print("Loaded file:", file_path)
print("Shape:", cp_processed.shape)
print("Columns:", cp_processed.columns.tolist())

print("\n" + "=" * 80)
print("UNIQUE DATA ITEMS")
print("=" * 80)

data_items = sorted(
    cp_processed["Data Item"]
    .dropna()
    .astype(str)
    .unique()
)

for i, item in enumerate(data_items, start=1):
    print(f"{i}. {item}")

print("\nTotal unique Data Items:", len(data_items))

print("\n" + "=" * 80)
print("POSSIBLE 5-YEAR AVERAGE DATA ITEMS")
print("=" * 80)

average_items = [
    item
    for item in data_items
    if "AVG" in item.upper()
    or "AVERAGE" in item.upper()
    or "5 YEAR" in item.upper()
]

for item in average_items:
    print(item)

print("\nTotal possible average Data Items:", len(average_items))

FILES FOUND
1. /Users/jason/Desktop/Intern/agri_dashboard/data/processed/cp_processed.csv

LOADING DATA
Loaded file: /Users/jason/Desktop/Intern/agri_dashboard/data/processed/cp_processed.csv
Shape: (172714, 21)
Columns: ['Program', 'Year', 'Period', 'Week Ending', 'Geo Level', 'State', 'State ANSI', 'Ag District', 'Ag District Code', 'County', 'County ANSI', 'Zip Code', 'Region', 'watershed_code', 'Watershed', 'Commodity', 'Data Item', 'Domain', 'Domain Category', 'Value', 'CV (%)']

UNIQUE DATA ITEMS
1. CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT DENTED
2. CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT DOUGH
3. CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT EMERGED
4. CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT MATURE
5. CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT MILK
6. CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT PLANTED
7. CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT SEEDBED PREPARED
8. CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT SILKING
9. CORN - PROGRESS, MEASURED IN PCT DENTED

In [79]:
import pandas as pd

debug_df = cp_processed.copy()

debug_df["Is 5Y Average"] = (
    debug_df["Data Item"]
    .astype("string")
    .str.contains("5 YEAR AVG", case=False, na=False)
)

debug_df["Week Ending"] = pd.to_datetime(
    debug_df["Week Ending"],
    errors="coerce",
)

debug_df["Week"] = debug_df["Week Ending"].dt.isocalendar().week

print("5-YEAR AVERAGE RECORDS")
print(
    debug_df.loc[
        debug_df["Is 5Y Average"],
        "Geo Level"
    ].value_counts(dropna=False)
)

print("\nCORN STATE 5-YEAR AVERAGE SAMPLE")

corn_avg = debug_df.loc[
    debug_df["Is 5Y Average"]
    & (debug_df["Commodity"] == "CORN")
    & (debug_df["Geo Level"] == "STATE")
]

print(
    corn_avg[
        [
            "Year",
            "Week Ending",
            "Week",
            "State",
            "Commodity",
            "Data Item",
            "Value",
        ]
    ].head(20).to_string(index=False)
)

print("\nAVERAGE RECORD COUNTS BY COMMODITY")

print(
    debug_df.loc[debug_df["Is 5Y Average"]]
    .groupby(["Commodity", "Geo Level"])
    .size()
)

print("\nAVAILABLE YEARS IN AVERAGE RECORDS")

print(
    sorted(
        debug_df.loc[
            debug_df["Is 5Y Average"],
            "Year"
        ].dropna().unique().tolist()
    )
)

5-YEAR AVERAGE RECORDS
Geo Level
STATE       96106
NATIONAL     3764
Name: count, dtype: int64

CORN STATE 5-YEAR AVERAGE SAMPLE
 Year Week Ending  Week       State Commodity                                            Data Item  Value
 2026  2026-03-01     9       TEXAS      CORN CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT PLANTED      7
 2026  2026-03-08    10       TEXAS      CORN CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT PLANTED     19
 2026  2026-03-15    11    ARKANSAS      CORN CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT PLANTED      1
 2026  2026-03-15    11   LOUISIANA      CORN CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT EMERGED     10
 2026  2026-03-15    11   LOUISIANA      CORN CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT PLANTED     34
 2026  2026-03-15    11 MISSISSIPPI      CORN CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT PLANTED      3
 2026  2026-03-15    11       TEXAS      CORN CORN - PROGRESS, 5 YEAR AVG, MEASURED IN PCT PLANTED     31
 2026  2026-03-22    12

In [80]:
normal = debug_df.loc[
    ~debug_df["Is 5Y Average"]
    & (debug_df["Commodity"] == "CORN")
    & (debug_df["Geo Level"] == "STATE")
].copy()

average = debug_df.loc[
    debug_df["Is 5Y Average"]
    & (debug_df["Commodity"] == "CORN")
    & (debug_df["Geo Level"] == "STATE")
].copy()

print("Normal progress weeks:")
print(sorted(normal["Week"].dropna().unique().tolist()))

print("\n5-year average weeks:")
print(sorted(average["Week"].dropna().unique().tolist()))

print("\nStates with 5-year averages:")
print(sorted(average["State"].dropna().unique().tolist()))

print("\nStates with normal progress:")
print(sorted(normal["State"].dropna().unique().tolist()))

print("\nSample normal progress records:")
print(
    normal[
        ["Year", "Week Ending", "State", "Data Item", "Value"]
    ].head(10).to_string(index=False)
)

Normal progress weeks:
[8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48]

5-year average weeks:
[8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48]

States with 5-year averages:
['ALABAMA', 'ARKANSAS', 'COLORADO', 'CONNECTICUT', 'DELAWARE', 'GEORGIA', 'IDAHO', 'ILLINOIS', 'INDIANA', 'IOWA', 'KANSAS', 'KENTUCKY', 'LOUISIANA', 'MAINE', 'MARYLAND', 'MASSACHUSETTS', 'MICHIGAN', 'MINNESOTA', 'MISSISSIPPI', 'MISSOURI', 'MONTANA', 'NEBRASKA', 'NEVADA', 'NEW HAMPSHIRE', 'NEW JERSEY', 'NEW MEXICO', 'NEW YORK', 'NORTH CAROLINA', 'NORTH DAKOTA', 'OHIO', 'OKLAHOMA', 'OREGON', 'PENNSYLVANIA', 'RHODE ISLAND', 'SOUTH CAROLINA', 'SOUTH DAKOTA', 'TENNESSEE', 'TEXAS', 'UTAH', 'VERMONT', 'VIRGINIA', 'WASHINGTON', 'WEST VIRGINIA', 'WISCONSIN', 'WYOMING']

States with normal progress:
[

In [81]:
import pandas as pd

commodity = "CORN"
measure = "PLANTED"
state = "IOWA"
week = 16
year = 2026

debug = cp_processed.copy()

debug["Week Ending"] = pd.to_datetime(
    debug["Week Ending"],
    errors="coerce",
)

debug["Week"] = (
    debug["Week Ending"]
    .dt.isocalendar()
    .week
    .astype("Int64")
)

debug["Is 5Y Average"] = (
    debug["Data Item"]
    .astype("string")
    .str.contains("5 YEAR AVG", case=False, na=False)
)

debug["Year"] = pd.to_numeric(
    debug["Year"],
    errors="coerce",
)

selected = debug.loc[
    (debug["Commodity"] == commodity)
    & (debug["Geo Level"] == "STATE")
    & (debug["State"] == state)
    & (debug["Week"] == week)
    & debug["Data Item"].str.contains(
        f"MEASURED IN PCT {measure}$",
        case=False,
        na=False,
    )
].copy()

print("MATCHING RECORDS")
print(
    selected[
        [
            "Year",
            "State",
            "Week",
            "Week Ending",
            "Data Item",
            "Value",
            "Is 5Y Average",
        ]
    ].to_string(index=False)
)

print("\nAVERAGE RECORD COUNT:")
print(selected["Is 5Y Average"].sum())

print("\nNORMAL RECORD COUNT:")
print((~selected["Is 5Y Average"]).sum())

MATCHING RECORDS
 Year State  Week Week Ending                                            Data Item  Value  Is 5Y Average
 2026  IOWA    16  2026-04-19             CORN - PROGRESS, MEASURED IN PCT PLANTED      2          False
 2025  IOWA    16  2025-04-20             CORN - PROGRESS, MEASURED IN PCT PLANTED     18          False
 2024  IOWA    16  2024-04-21             CORN - PROGRESS, MEASURED IN PCT PLANTED     13          False
 2023  IOWA    16  2023-04-23             CORN - PROGRESS, MEASURED IN PCT PLANTED     10          False
 2022  IOWA    16  2022-04-24             CORN - PROGRESS, MEASURED IN PCT PLANTED      2          False
 2021  IOWA    16  2021-04-25             CORN - PROGRESS, MEASURED IN PCT PLANTED     20          False
 2020  IOWA    16  2020-04-19             CORN - PROGRESS, MEASURED IN PCT PLANTED      2          False
 2019  IOWA    16  2019-04-21             CORN - PROGRESS, MEASURED IN PCT PLANTED      4          False
 2018  IOWA    16  2018-04-22         

In [83]:
from pathlib import Path
import pandas as pd

file_path = Path("../data/processed/cc_processed.csv")

cc = pd.read_csv(
    file_path,
    low_memory=False,
)

print("Shape:", cc.shape)

print("\nColumns:")
print(cc.columns.tolist())

display(cc.head(10))

Shape: (263238, 21)

Columns:
['Program', 'Year', 'Period', 'Week Ending', 'Geo Level', 'State', 'State ANSI', 'Ag District', 'Ag District Code', 'County', 'County ANSI', 'Zip Code', 'Region', 'watershed_code', 'Watershed', 'Commodity', 'Data Item', 'Domain', 'Domain Category', 'Value', 'CV (%)']


,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Zip Code,Region,watershed_code,Watershed,Commodity,Data Item,Domain,Domain Category,Value,CV (%)
0,SURVEY,2026,WEEK #14,2026-04-05,STATE,LOUISIANA,22.0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,CORN,"CORN - CONDITION, MEASURED IN PCT EXCELLENT",TOTAL,NOT SPECIFIED,0,NaN
1,SURVEY,2026,WEEK #14,2026-04-05,STATE,LOUISIANA,22.0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,CORN,"CORN - CONDITION, MEASURED IN PCT FAIR",TOTAL,NOT SPECIFIED,36,NaN
2,SURVEY,2026,WEEK #14,2026-04-05,STATE,LOUISIANA,22.0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,CORN,"CORN - CONDITION, MEASURED IN PCT GOOD",TOTAL,NOT SPECIFIED,54,NaN
3,SURVEY,2026,WEEK #14,2026-04-05,STATE,LOUISIANA,22.0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,CORN,"CORN - CONDITION, MEASURED IN PCT POOR",TOTAL,NOT SPECIFIED,10,NaN
4,SURVEY,2026,WEEK #14,2026-04-05,STATE,LOUISIANA,22.0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,CORN,"CORN - CONDITION, MEASURED IN PCT VERY POOR",TOTAL,NOT SPECIFIED,0,NaN
5,SURVEY,2026,WEEK #15,2026-04-12,STATE,LOUISIANA,22.0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,CORN,"CORN - CONDITION, MEASURED IN PCT EXCELLENT",TOTAL,NOT SPECIFIED,0,NaN
6,SURVEY,2026,WEEK #15,2026-04-12,STATE,LOUISIANA,22.0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,CORN,"CORN - CONDITION, MEASURED IN PCT FAIR",TOTAL,NOT SPECIFIED,30,NaN
7,SURVEY,2026,WEEK #15,2026-04-12,STATE,LOUISIANA,22.0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,CORN,"CORN - CONDITION, MEASURED IN PCT GOOD",TOTAL,NOT SPECIFIED,59,NaN
8,SURVEY,2026,WEEK #15,2026-04-12,STATE,LOUISIANA,22.0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,CORN,"CORN - CONDITION, MEASURED IN PCT POOR",TOTAL,NOT SPECIFIED,11,NaN
9,SURVEY,2026,WEEK #15,2026-04-12,STATE,LOUISIANA,22.0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,CORN,"CORN - CONDITION, MEASURED IN PCT VERY POOR",TOTAL,NOT SPECIFIED,0,NaN


In [84]:
print("DATA TYPES")
display(cc.dtypes)

print("\nUNIQUE COMMODITIES")
display(cc["Commodity"].value_counts(dropna=False))

print("\nGEOGRAPHICAL LEVELS")
display(cc["Geo Level"].value_counts(dropna=False))

print("\nPROGRAMS")
display(cc["Program"].value_counts(dropna=False))

print("\nDATA ITEM EXAMPLES")
display(
    cc["Data Item"]
    .value_counts()
    .head(50)
    .to_frame("Count")
)

DATA TYPES


Program                 str
Year                  int64
Period                  str
Week Ending             str
Geo Level               str
State                   str
State ANSI          float64
Ag District         float64
Ag District Code    float64
County              float64
County ANSI         float64
Zip Code            float64
Region              float64
watershed_code        int64
Watershed           float64
Commodity               str
Data Item               str
Domain                  str
Domain Category         str
Value                 int64
CV (%)              float64
dtype: object


UNIQUE COMMODITIES


Commodity
WHEAT       95536
CORN        95412
SOYBEANS    72290
Name: count, dtype: int64


GEOGRAPHICAL LEVELS


Geo Level
STATE       255358
NATIONAL      7880
Name: count, dtype: int64


PROGRAMS


Program
SURVEY    263238
Name: count, dtype: int64


DATA ITEM EXAMPLES


,Count
Data Item,
"CORN - CONDITION, 5 YEAR AVG, MEASURED IN PCT EXCELLENT",9811
"CORN - CONDITION, 5 YEAR AVG, MEASURED IN PCT FAIR",9811
"CORN - CONDITION, 5 YEAR AVG, MEASURED IN PCT GOOD",9811
"CORN - CONDITION, 5 YEAR AVG, MEASURED IN PCT POOR",9810
"CORN - CONDITION, 5 YEAR AVG, MEASURED IN PCT VERY POOR",9739
"CORN - CONDITION, MEASURED IN PCT EXCELLENT",9286
"CORN - CONDITION, MEASURED IN PCT FAIR",9286
"CORN - CONDITION, MEASURED IN PCT GOOD",9286
"CORN - CONDITION, MEASURED IN PCT POOR",9286


In [ ]:
keys = [
    "Year",
    "Period",
    "Week Ending",
    "State",
    "Commodity",
    "Crop Type",
    "Is 5Y Average",
]

state_data = data.loc[
    data["Geo Level"].astype(str).str.upper().eq("STATE")
].copy()

category_check = (
    state_data.groupby(keys, dropna=False)
    .agg(
        Category_Count=("Condition", "nunique"),
        Value_Count=("Value", "count"),
        Percentage_Total=("Value", "sum"),
    )
    .reset_index()
)

print("1. CATEGORY COUNTS")
print(category_check["Category_Count"].value_counts().sort_index())

print("\n2. NUMBER OF GROUPS WITH TOTALS OTHER THAN 100")
print(
    category_check.loc[
        category_check["Percentage_Total"] != 100
    ].shape[0]
)

print("\n3. TOTALS ABOVE 101")
display(
    category_check.loc[
        category_check["Percentage_Total"] > 101
    ]
)

print("\n4. INCOMPLETE CATEGORY GROUPS")
display(
    category_check.loc[
        category_check["Category_Count"] != 5
    ]
)

CATEGORY COMPLETENESS BY RECORD TYPE


Category_Count,1,2,3,4,5
Is 5Y Average,,,,,
False,0,0,0,0,25030
True,32,2,1,216,25861



INCOMPLETE ACTUAL RECORDS


,Year,Period,Week Ending,State,Commodity,Crop Type,Is 5Y Average,Category_Count,Value_Count,Percentage_Total



INCOMPLETE 5-YEAR AVERAGE RECORDS


Groups
Year Crop Type                  Category_Count        
2014 Corn                       1                    7
                                3                    1
                                4                    8
     Soybeans                   1                    8
                                4                    7
     Spring Durum Wheat         1                    2
     Spring Wheat (Excl. Durum) 1                    5
                                2                    2
     Winter Wheat               1                   10
                                4                   10
2015 Corn                       4                   16
     Soybeans                   4                   16
     Winter Wheat               4                   12
2016 Corn                       4                   19
     Soybeans                   4                   17
     Winter Wheat               4                   11
2017 Corn                       4                   19
     Soybeans                   4                   19
     Winter Wheat               4                   13
2018 Corn                       4                   16
     Soybeans                   4                   19
     Winter Wheat               4                   14

In [93]:
print("CATEGORY COMPLETENESS BY RECORD TYPE")

display(
    category_check.groupby("Is 5Y Average")["Category_Count"]
    .value_counts()
    .unstack(fill_value=0)
)

print("\nINCOMPLETE ACTUAL RECORDS")

display(
    category_check.loc[
        (~category_check["Is 5Y Average"])
        & (category_check["Category_Count"] != 5)
    ]
)

print("\nINCOMPLETE 5-YEAR AVERAGE RECORDS")

display(
    category_check.loc[
        category_check["Is 5Y Average"]
        & (category_check["Category_Count"] != 5)
    ].groupby(
        ["Year", "Crop Type", "Category_Count"]
    ).size().to_frame("Groups")
)

CATEGORY COMPLETENESS BY RECORD TYPE


Category_Count,1,2,3,4,5
Is 5Y Average,,,,,
False,0,0,0,0,25030
True,32,2,1,216,25861



INCOMPLETE ACTUAL RECORDS


,Year,Period,Week Ending,State,Commodity,Crop Type,Is 5Y Average,Category_Count,Value_Count,Percentage_Total



INCOMPLETE 5-YEAR AVERAGE RECORDS


Groups
Year Crop Type                  Category_Count        
2014 Corn                       1                    7
                                3                    1
                                4                    8
     Soybeans                   1                    8
                                4                    7
     Spring Durum Wheat         1                    2
     Spring Wheat (Excl. Durum) 1                    5
                                2                    2
     Winter Wheat               1                   10
                                4                   10
2015 Corn                       4                   16
     Soybeans                   4                   16
     Winter Wheat               4                   12
2016 Corn                       4                   19
     Soybeans                   4                   17
     Winter Wheat               4                   11
2017 Corn                       4                   19
     Soybeans                   4                   19
     Winter Wheat               4                   13
2018 Corn                       4                   16
     Soybeans                   4                   19
     Winter Wheat               4                   14

In [97]:
from src.transforms.transform_cc import prepare_cc_data

national_cc, state_cc, state_table_cc = prepare_cc_data(cc)

In [ ]:
from src.transforms.transform_cc import (
    is_five_year_average,
    prepare_cc_base_data,
    prepare_cc_data,
)

# Prepare all three datasets
national_cc, state_cc, state_table_cc = prepare_cc_data(cc)

# 1. Compare source and transformed row counts
source_actual = cc.loc[~is_five_year_average(cc)].copy()
base_actual = prepare_cc_base_data(cc)

print("SOURCE ACTUAL ROWS:", len(source_actual))
print("TRANSFORMED ROWS:", len(base_actual))
print("NATIONAL ROWS:", len(national_cc))
print("STATE ROWS:", len(state_cc))

print("\nSOURCE GEOGRAPHY COUNTS")
display(source_actual["Geo Level"].value_counts(dropna=False))

print("\nTRANSFORMED GEOGRAPHY COUNTS")
display(base_actual["Geo Level"].value_counts(dropna=False))


# 2. Check missing derived fields
print("\nMISSING CROP TYPE")
display(
    base_actual.loc[
        base_actual["Crop Type"].isna(),
        ["Commodity", "Data Item"],
    ].drop_duplicates()
)

print("\nMISSING CONDITION")
display(
    base_actual.loc[
        base_actual["Condition"].isna(),
        ["Commodity", "Data Item"],
    ].drop_duplicates()
)

print("\nMISSING WEEK")
display(
    base_actual.loc[
        base_actual["Week"].isna(),
        ["Year", "Period", "Data Item"],
    ].head(20)
)


# 3. Check category completeness: National
national_check = (
    national_cc.groupby(
        ["Year", "Week", "Commodity", "Crop Type"],
        observed=True,
        dropna=False,
    )
    .agg(
        Category_Count=("Condition", "nunique"),
        Percentage_Total=("Value", "sum"),
    )
    .reset_index()
)

print("\nNATIONAL CATEGORY COUNTS")
display(
    national_check["Category_Count"]
    .value_counts()
    .sort_index()
)

print("\nNATIONAL GROUPS WITH FEWER THAN 5 CATEGORIES")
display(
    national_check.loc[
        national_check["Category_Count"] != 5
    ].head(30)
)

print("\nNATIONAL TOTALS NOT EQUAL TO 100")
display(
    national_check.loc[
        national_check["Percentage_Total"] != 100
    ].head(30)
)


# 4. Check category completeness: State
state_check = (
    state_cc.groupby(
        ["Year", "Week", "State", "Commodity", "Crop Type"],
        observed=True,
        dropna=False,
    )
    .agg(
        Category_Count=("Condition", "nunique"),
        Percentage_Total=("Value", "sum"),
    )
    .reset_index()
)

print("\nSTATE CATEGORY COUNTS")
display(
    state_check["Category_Count"]
    .value_counts()
    .sort_index()
)

print("\nSTATE GROUPS WITH FEWER THAN 5 CATEGORIES")
display(
    state_check.loc[
        state_check["Category_Count"] != 5
    ].head(30)
)

print("\nSTATE TOTALS NOT EQUAL TO 100")
display(
    state_check.loc[
        state_check["Percentage_Total"] != 100
    ].head(30)
)


# 5. Check duplicate observations
duplicate_keys = [
    "Year",
    "Week",
    "Week Ending",
    "Geo Level",
    "State",
    "Commodity",
    "Crop Type",
    "Condition",
]

print("\nDUPLICATE NATIONAL OBSERVATIONS")
display(
    national_cc.loc[
        national_cc.duplicated(
            subset=[
                "Year", "Week", "Commodity",
                "Crop Type", "Condition",
            ],
            keep=False,
        )
    ].sort_values(
        ["Year", "Week", "Commodity", "Crop Type", "Condition"]
    ).head(30)
)

print("\nDUPLICATE STATE OBSERVATIONS")
display(
    state_cc.loc[
        state_cc.duplicated(
            subset=duplicate_keys,
            keep=False,
        )
    ].sort_values(
        ["Year", "Week", "State", "Commodity", "Crop Type", "Condition"]
    ).head(30)
)

SOURCE ACTUAL ROWS: 128720
TRANSFORMED ROWS: 128720
NATIONAL ROWS: 3570
STATE ROWS: 125150

SOURCE GEOGRAPHY COUNTS


Geo Level
STATE       125150
NATIONAL      3570
Name: count, dtype: int64


TRANSFORMED GEOGRAPHY COUNTS


Geo Level
STATE       125150
NATIONAL      3570
Name: count, dtype: int64[pyarrow]


MISSING CROP TYPE


,Commodity,Data Item


... (16 outputs left out)


DUPLICATE NATIONAL OBSERVATIONS


,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Commodity,Data Item,Domain,Domain Category,Value,CV (%),Week,Crop Type Raw,Crop Type,Condition



DUPLICATE STATE OBSERVATIONS


,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Commodity,Data Item,Domain,Domain Category,Value,CV (%),Week,Crop Type Raw,Crop Type,Condition


In [104]:
from pathlib import Path
import pandas as pd

file_path = Path("../data/processed/cp_processed.csv")
cp = pd.read_csv(file_path)

print("Loaded:", cp.shape)

Loaded: (172714, 21)


In [105]:
from src.transforms.transform_cp import prepare_cp_base_data

cp_actual = cp.loc[
    ~cp["Data Item"].astype("string").str.contains(
        r"5[\s-]*YEAR[\s-]*AVG",
        case=False,
        na=False,
    )
].copy()

cp_base = prepare_cp_base_data(cp_actual)

print("Source actual rows:", len(cp_actual))
print("Transformed rows:", len(cp_base))

Source actual rows: 72844
Transformed rows: 72844


In [109]:
import pandas as pd

from src.transforms.transform_cp import prepare_cp_base_data


cp = pd.read_csv("../data/processed/cp_processed.csv")

cp_actual = cp.loc[
    ~cp["Data Item"].astype("string").str.contains(
        r"5[\s-]*YEAR[\s-]*AVG",
        case=False,
        na=False,
    )
].copy()

cp_base = prepare_cp_base_data(cp_actual)


print("=" * 70)
print("1. SOURCE AND TRANSFORMED ROW COUNTS")
print("=" * 70)

print("Source actual rows:", len(cp_actual))
print("Transformed actual rows:", len(cp_base))
print("Rows preserved:", len(cp_actual) == len(cp_base))

print("\nGEOGRAPHY COUNTS")
display(cp_base["Geo Level"].value_counts(dropna=False))


print("\n" + "=" * 70)
print("2. MULTIPLE WEEK ENDING DATES")
print("=" * 70)

date_keys = [
    "Geo Level",
    "Year",
    "Week",
    "Commodity",
    "Crop Type",
    "Measure",
    "State",
]

cp_date_check = (
    cp_base.groupby(date_keys, dropna=False)
    .agg(
        Date_Count=("Week Ending", "nunique"),
        Row_Count=("Value", "count"),
        Week_Ending_Dates=(
            "Week Ending",
            lambda x: sorted(x.dropna().unique()),
        ),
    )
    .reset_index()
)

multiple_dates = cp_date_check.loc[
    cp_date_check["Date_Count"] > 1
].copy()

print("Groups with multiple Week Ending dates:", len(multiple_dates))
display(multiple_dates.head(50))


print("\n" + "=" * 70)
print("3. SEASONAL DATA DUPLICATES")
print("=" * 70)

seasonal_keys = [
    "Geo Level",
    "Commodity",
    "Crop Type",
    "Measure",
    "Year",
    "Week",
    "State",
]

seasonal_duplicates = cp_base.loc[
    cp_base.duplicated(subset=seasonal_keys, keep=False)
].copy()

print("Rows involved in seasonal duplicates:", len(seasonal_duplicates))

display(
    seasonal_duplicates.sort_values(
        seasonal_keys + ["Week Ending"]
    ).head(50)
)


print("\n" + "=" * 70)
print("4. NATIONAL WEEKLY COMPARISON DUPLICATES")
print("=" * 70)

national_comparison_keys = [
    "Commodity",
    "Crop Type",
    "Measure",
    "Year",
    "Week",
]

national_duplicates = cp_base.loc[
    (cp_base["Geo Level"] == "NATIONAL")
    & cp_base.duplicated(
        subset=national_comparison_keys,
        keep=False,
    )
].copy()

print(
    "Rows involved in national weekly comparison duplicates:",
    len(national_duplicates),
)

display(
    national_duplicates.sort_values(
        national_comparison_keys + ["Week Ending"]
    ).head(50)
)


print("\n" + "=" * 70)
print("5. STATE WEEKLY DATA DUPLICATES")
print("=" * 70)

state_comparison_keys = [
    "Commodity",
    "Crop Type",
    "Measure",
    "Year",
    "Week",
    "State",
]

state_duplicates = cp_base.loc[
    (cp_base["Geo Level"] == "STATE")
    & cp_base.duplicated(
        subset=state_comparison_keys,
        keep=False,
    )
].copy()

print("Rows involved in state weekly duplicates:", len(state_duplicates))

display(
    state_duplicates.sort_values(
        state_comparison_keys + ["Week Ending"]
    ).head(50)
)


print("\n" + "=" * 70)
print("6. EXACT DUPLICATE ROWS")
print("=" * 70)

exact_duplicates = cp_base.loc[
    cp_base.duplicated(keep=False)
].copy()

print("Exact duplicate rows:", len(exact_duplicates))
display(exact_duplicates.head(50))


print("\n" + "=" * 70)
print("7. MISSING DERIVED FIELDS")
print("=" * 70)

for column in [
    "Year",
    "Week",
    "Week Ending",
    "Crop Type",
    "Measure",
    "Value",
]:
    print(f"{column}: {cp_base[column].isna().sum()}")


print("\n" + "=" * 70)
print("8. DIAGNOSTIC SUMMARY")
print("=" * 70)

print("Multiple reporting-date groups:", len(multiple_dates))
print("Seasonal duplicate rows:", len(seasonal_duplicates))
print("National comparison duplicate rows:", len(national_duplicates))
print("State comparison duplicate rows:", len(state_duplicates))
print("Exact duplicate rows:", len(exact_duplicates))
print("Source rows preserved:", len(cp_actual) == len(cp_base))

1. SOURCE AND TRANSFORMED ROW COUNTS
Source actual rows: 72844
Transformed actual rows: 72844
Rows preserved: True

GEOGRAPHY COUNTS


Geo Level
STATE       70836
NATIONAL     2008
Name: count, dtype: int64[pyarrow]


2. MULTIPLE WEEK ENDING DATES
Groups with multiple Week Ending dates: 18


,Geo Level,Year,Week,Commodity,Crop Type,Measure,State,Date_Count,Row_Count,Week_Ending_Dates
52184,STATE,2023,40,WHEAT,Winter Wheat,Emerged,NEW MEXICO,2,2,"[2022-10-09 00:00:00, 2023-10-08 00:00:00]"
52352,STATE,2023,41,WHEAT,Winter Wheat,Emerged,NEW MEXICO,2,2,"[2022-10-16 00:00:00, 2023-10-15 00:00:00]"
53055,STATE,2023,52,WHEAT,Winter Wheat,Pastured,OKLAHOMA,2,2,"[2023-01-01 00:00:00, 2023-12-31 00:00:00]"
58208,STATE,2024,36,WHEAT,Winter Wheat,Planted,NEW MEXICO,2,2,"[2023-09-10 00:00:00, 2024-09-08 00:00:00]"
58414,STATE,2024,37,WHEAT,Winter Wheat,Planted,NEW MEXICO,2,2,"[2023-09-17 00:00:00, 2024-09-15 00:00:00]"
58619,STATE,2024,38,WHEAT,Winter Wheat,Planted,NEW MEXICO,2,2,"[2023-09-24 00:00:00, 2024-09-22 00:00:00]"
65193,STATE,2025,37,WHEAT,Winter Wheat,Planted,MISSISSIPPI,2,2,"[2024-09-15 00:00:00, 2025-09-14 00:00:00]"
65205,STATE,2025,37,WHEAT,Winter Wheat,Planted,TENNESSEE,2,2,"[2024-09-15 00:00:00, 2025-09-14 00:00:00]"
65420,STATE,2025,38,WHEAT,Winter Wheat,Planted,TENNESSEE,2,2,"[2024-09-22 00:00:00, 2025-09-21 00:00:00]"
65569,STATE,2025,39,WHEAT,Winter Wheat,Emerged,MISSISSIPPI,2,2,"[2024-09-29 00:00:00, 2025-09-28 00:00:00]"



3. SEASONAL DATA DUPLICATES
Rows involved in seasonal duplicates: 36


,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Watershed,Commodity,Data Item,Domain,Domain Category,Value,CV (%),Week,Crop Type,Measure
120191,SURVEY,2023,WEEK #40,2022-10-09,STATE,NEW MEXICO,35.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,45,NaN,40,Winter Wheat,Emerged
120224,SURVEY,2023,WEEK #40,2023-10-08,STATE,NEW MEXICO,35.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,47,NaN,40,Winter Wheat,Emerged
120261,SURVEY,2023,WEEK #41,2022-10-16,STATE,NEW MEXICO,35.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,63,NaN,41,Winter Wheat,Emerged
120294,SURVEY,2023,WEEK #41,2023-10-15,STATE,NEW MEXICO,35.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,62,NaN,41,Winter Wheat,Emerged
115122,SURVEY,2025,WEEK #39,2024-09-29,STATE,MISSISSIPPI,28.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,1,NaN,39,Winter Wheat,Emerged
115164,SURVEY,2025,WEEK #39,2025-09-28,STATE,MISSISSIPPI,28.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,0,NaN,39,Winter Wheat,Emerged
115643,SURVEY,2025,WEEK #46,2024-11-17,STATE,TENNESSEE,47.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,69,NaN,46,Winter Wheat,Emerged
115653,SURVEY,2025,WEEK #46,2025-11-16,STATE,TENNESSEE,47.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,59,NaN,46,Winter Wheat,Emerged
115708,SURVEY,2025,WEEK #47,2024-11-24,STATE,TENNESSEE,47.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,80,NaN,47,Winter Wheat,Emerged
115718,SURVEY,2025,WEEK #47,2025-11-23,STATE,TENNESSEE,47.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,65,NaN,47,Winter Wheat,Emerged


... (3 outputs left out)

,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Watershed,Commodity,Data Item,Domain,Domain Category,Value,CV (%),Week,Crop Type,Measure
120191,SURVEY,2023,WEEK #40,2022-10-09,STATE,NEW MEXICO,35.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,45,NaN,40,Winter Wheat,Emerged
120224,SURVEY,2023,WEEK #40,2023-10-08,STATE,NEW MEXICO,35.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,47,NaN,40,Winter Wheat,Emerged
120261,SURVEY,2023,WEEK #41,2022-10-16,STATE,NEW MEXICO,35.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,63,NaN,41,Winter Wheat,Emerged
120294,SURVEY,2023,WEEK #41,2023-10-15,STATE,NEW MEXICO,35.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,62,NaN,41,Winter Wheat,Emerged
115122,SURVEY,2025,WEEK #39,2024-09-29,STATE,MISSISSIPPI,28.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,1,NaN,39,Winter Wheat,Emerged
115164,SURVEY,2025,WEEK #39,2025-09-28,STATE,MISSISSIPPI,28.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,0,NaN,39,Winter Wheat,Emerged
115643,SURVEY,2025,WEEK #46,2024-11-17,STATE,TENNESSEE,47.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,69,NaN,46,Winter Wheat,Emerged
115653,SURVEY,2025,WEEK #46,2025-11-16,STATE,TENNESSEE,47.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,59,NaN,46,Winter Wheat,Emerged
115708,SURVEY,2025,WEEK #47,2024-11-24,STATE,TENNESSEE,47.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,80,NaN,47,Winter Wheat,Emerged
115718,SURVEY,2025,WEEK #47,2025-11-23,STATE,TENNESSEE,47.0,NaN,NaN,NaN,...,NaN,WHEAT,"WHEAT, WINTER - PROGRESS, MEASURED IN PCT EMERGED",TOTAL,NOT SPECIFIED,65,NaN,47,Winter Wheat,Emerged



6. EXACT DUPLICATE ROWS
Exact duplicate rows: 0


,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Watershed,Commodity,Data Item,Domain,Domain Category,Value,CV (%),Week,Crop Type,Measure



7. MISSING DERIVED FIELDS
Year: 0
Week: 0
Week Ending: 0
Crop Type: 0
Measure: 0
Value: 0

8. DIAGNOSTIC SUMMARY
Multiple reporting-date groups: 18
Seasonal duplicate rows: 36
National comparison duplicate rows: 2008
State comparison duplicate rows: 36
Exact duplicate rows: 0
Source rows preserved: True


In [110]:
print("=" * 70)
print("1. NATIONAL DUPLICATE GROUP COUNTS")
print("=" * 70)

national_group_counts = (
    cp_base.loc[cp_base["Geo Level"] == "NATIONAL"]
    .groupby(
        ["Commodity", "Crop Type", "Measure", "Year", "Week"],
        dropna=False,
    )
    .size()
    .reset_index(name="Row Count")
)

duplicate_groups = national_group_counts.loc[
    national_group_counts["Row Count"] > 1
].copy()

print("National rows:", len(cp_base.loc[cp_base["Geo Level"] == "NATIONAL"]))
print("Duplicate groups:", len(duplicate_groups))
print("Rows in duplicate groups:", duplicate_groups["Row Count"].sum())
print("\nRows per group:")
display(duplicate_groups["Row Count"].value_counts().sort_index())


print("\n" + "=" * 70)
print("2. FIELDS THAT DIFFER WITHIN DUPLICATE GROUPS")
print("=" * 70)

national_keys = [
    "Commodity", "Crop Type", "Measure", "Year", "Week"
]

national_check = cp_base.loc[
    cp_base["Geo Level"] == "NATIONAL"
].copy()

other_columns = [
    col for col in national_check.columns
    if col not in national_keys
]

difference_summary = []

for keys, group in national_check.groupby(national_keys, dropna=False):
    if len(group) > 1:
        for col in other_columns:
            unique_count = group[col].nunique(dropna=False)
            if unique_count > 1:
                difference_summary.append({
                    "Duplicate Key": keys,
                    "Field": col,
                    "Unique Values": unique_count,
                    "Values": group[col].astype(str).unique().tolist(),
                })

difference_df = pd.DataFrame(difference_summary)

print("Fields with differing values:", difference_df["Field"].nunique())
display(
    difference_df.groupby("Field")
    .size()
    .reset_index(name="Groups Where Field Differs")
    .sort_values("Groups Where Field Differs", ascending=False)
)


print("\n" + "=" * 70)
print("3. SAMPLE NATIONAL DUPLICATE RECORDS")
print("=" * 70)

if not duplicate_groups.empty:
    sample_key = duplicate_groups.iloc[0][national_keys].tolist()

    sample = national_check.copy()

    for col, value in zip(national_keys, sample_key):
        sample = sample.loc[sample[col].eq(value)]

    display(sample.T)


print("\n" + "=" * 70)
print("4. SOURCE DATA ITEM AND DOMAIN BREAKDOWN")
print("=" * 70)

display(
    national_check.loc[
        national_check.duplicated(national_keys, keep=False),
        [
            "Year",
            "Week",
            "Week Ending",
            "Commodity",
            "Crop Type",
            "Measure",
            "Data Item",
            "Domain",
            "Domain Category",
            "Value",
        ],
    ].sort_values(national_keys + ["Week Ending"])
    .head(50)
)

1. NATIONAL DUPLICATE GROUP COUNTS
National rows: 2008
Duplicate groups: 0
Rows in duplicate groups: 0

Rows per group:


Series([], Name: count, dtype: int64)


2. FIELDS THAT DIFFER WITHIN DUPLICATE GROUPS


KeyError: 'Field'

In [ ]:
from pathlib import Path
import pandas as pd



/var/folders/x6/jwk_yvpj1j58bf5kswkfwl9w0000gn/T/ipykernel_17878/2092107190.py:5: DtypeWarning: Columns (120: traders_prod_merc_long_all) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/cftc_agri_011126.csv")
